# Replicação Fiel do Artigo: VGG-16 Treinada do Zero (weights=None)
Este notebook executa o treinamento da CNN **VGG-16** inicializada do zero (`weights=None`), exatamente como na metodologia do artigo científico:
> *"Preprocessing and Optimizer Impact on Image-Based Autoimmune Disease Diagnosis"* (RITA, 2025).

**Objetivo:** Replicar a Tabela 3 na imagem **Original**, demonstrando o contraste entre a convergência com **SGD** e o colapso com **Adam**.

### 1. Verificar GPU e Clonar Repositório
Certifique-se de que o ambiente está com GPU ativa (**Ambiente de Execução > Alterar tipo de ambiente de execução > T4 GPU**).

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.exists("/content/Projeto-PDI"):
    !git clone -b feat/vgg-16 https://github.com/orleoncio/Projeto-PDI.git /content/Projeto-PDI

%cd /content/Projeto-PDI
!git checkout feat/vgg-16
!git pull origin feat/vgg-16

### 2. Instalar Dependências

In [ ]:
!pip install -q scikit-learn pandas pillow torchvision torch

### 3. Validar a Arquitetura da VGG-16 (16 camadas, 6 classes)

In [ ]:
%cd /content/Projeto-PDI
!python -u tests/test_vgg16.py

### 4. Treinamento da VGG-16 do Zero com SGD
* **Configuração:** `weights=None` (sem pesos pré-treinados), 30 épocas, batch size 32.
* **Meta do artigo (Tabela 3):** ~98,16% de acurácia.
* Salva os pesos em `results/vgg16_original_sgd.pth`.

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer sgd --epochs 30 --batch-size 32 --amp --num-workers 2

### 5. Treinamento da VGG-16 do Zero com Adam
* **Configuração:** `weights=None`, 30 épocas, batch size 32.
* **Comportamento esperado do artigo (Tabela 3):** Colapso para ~19,96% de acurácia (mínimo local).
* Salva os pesos em `results/vgg16_original_adam.pth`.

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer adam --epochs 30 --batch-size 32 --amp --num-workers 2

### 6. Confrontação Direta com a Tabela 3 do Artigo
Carrega os dois modelos treinados do zero e monta a tabela comparativa oficial.

In [ ]:
import torch
import pandas as pd
from data.hep2_dataset import create_dataloaders
from models.vgg16 import create_vgg16
from training.test import test_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, test_loader = create_dataloaders(batch_size=32, normalize=False)

# Valores reportados na Tabela 3 do artigo para a VGG-16 na imagem Original (treinada do zero)
paper_metrics = {
    "sgd": {"accuracy": 0.9816, "f1": 0.9805, "precision": 0.9814, "recall": 0.9798},
    "adam": {"accuracy": 0.1996, "f1": 0.0555, "precision": 0.0333, "recall": 0.1667}
}

comparison_rows = []

for opt in ["sgd", "adam"]:
    path = f"results/vgg16_original_{opt}.pth"
    try:
        model = create_vgg16(num_classes=6, pretrained=False).to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        print(f"\n>>> Avaliando VGG-16 do Zero ({opt.upper()}) <<<")
        res = test_model(model, test_loader, device)
        
        for metric_name in ["accuracy", "f1", "precision", "recall"]:
            val_obtained = res[metric_name] * 100
            val_paper = paper_metrics[opt][metric_name] * 100
            diff = val_obtained - val_paper
            comparison_rows.append({
                "Otimizador": opt.upper(),
                "Métrica": metric_name.capitalize(),
                "Artigo (%)": f"{val_paper:.2f}%",
                "Nosso Treino (%)": f"{val_obtained:.2f}%",
                "Diferença (%)": f"{diff:+.2f}%"
            })
    except FileNotFoundError:
        print(f"Arquivo de pesos {path} ainda não foi gerado. Execute a célula correspondente.")

if comparison_rows:
    df_comp = pd.DataFrame(comparison_rows)
    print("\n=================== TABELA COMPARATIVA VGG-16 DO ZERO (ORIGINAL) ===================")
    display(df_comp)
